In [1]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 127.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 159.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 164.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 185.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 169.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 MB 118.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.2/188.2 MB 175.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [2]:
!pip install -U typing_extensions


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
from unsloth import FastLanguageModel
import torch


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.1-8B-Instruct",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    token = "", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.25 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0.05, # Supports any, but = 0 is optimized
    bias = "lora_only",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth: bias = `none` is supported for fast patching. You are using bias = lora_only.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.2 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [3]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

In [16]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="./final_dataset.sft.jsonl",
    split="train"
)

print(dataset)
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 8000
})
{'messages': [{'role': 'system', 'content': '\nYou are an expert cybersecurity email classification system.\n\nYour task is to analyze emails and classify them into EXACTLY one of the following categories:\n\n- valid\n- spam\n- phishing\n\nCategory definitions:\n\nvalid:\nLegitimate personal, business, enterprise, transactional, operational, or opt-in communication from real organizations or individuals.\n\nspam:\nUnsolicited junk, low-quality advertising, irrelevant mass messaging, generic scams, or misleading promotional content that does NOT primarily attempt credential theft, impersonation, or account compromise.\n\nphishing:\nMalicious or deceptive emails attempting credential theft, impersonation, account compromise, malware delivery, financial fraud, payment fraud, social engineering, or unauthorized access.\n\nIf an email contains suspicious wording but lacks clear malicious intent, credential theft, impersonation, or 

In [17]:
from unsloth.chat_templates import standardize_data_formats

dataset = standardize_data_formats(dataset)

In [18]:
dataset[100]

{'messages': [{'role': 'system',
   'content': '\nYou are an expert cybersecurity email classification system.\n\nYour task is to analyze emails and classify them into EXACTLY one of the following categories:\n\n- valid\n- spam\n- phishing\n\nCategory definitions:\n\nvalid:\nLegitimate personal, business, enterprise, transactional, operational, or opt-in communication from real organizations or individuals.\n\nspam:\nUnsolicited junk, low-quality advertising, irrelevant mass messaging, generic scams, or misleading promotional content that does NOT primarily attempt credential theft, impersonation, or account compromise.\n\nphishing:\nMalicious or deceptive emails attempting credential theft, impersonation, account compromise, malware delivery, financial fraud, payment fraud, social engineering, or unauthorized access.\n\nIf an email contains suspicious wording but lacks clear malicious intent, credential theft, impersonation, or fraud indicators, classify it as spam instead of phishing

In [19]:
# Shuffle dataset
dataset = dataset.shuffle(seed=3407)

# Train / eval split
split_dataset = dataset.train_test_split(
    test_size=0.1,
    seed=3407
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [20]:
def formatting_prompts_func(examples):
    texts = []

    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        text += tokenizer.eos_token
        texts.append(text)

    return {"text": texts}

In [21]:
train_dataset = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=train_dataset.column_names,
)

eval_dataset = eval_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=eval_dataset.column_names,
)

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [22]:
print(train_dataset.column_names)
print(train_dataset[0]["text"][:1000])

['text']
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024


You are an expert cybersecurity email classification system.

Your task is to analyze emails and classify them into EXACTLY one of the following categories:

- valid
- spam
- phishing

Category definitions:

valid:
Legitimate personal, business, enterprise, transactional, operational, or opt-in communication from real organizations or individuals.

spam:
Unsolicited junk, low-quality advertising, irrelevant mass messaging, generic scams, or misleading promotional content that does NOT primarily attempt credential theft, impersonation, or account compromise.

phishing:
Malicious or deceptive emails attempting credential theft, impersonation, account compromise, malware delivery, financial fraud, payment fraud, social engineering, or unauthorized access.

If an email contains suspicious wording but lacks clear malicious intent, credential theft, impersona

In [25]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    args=SFTConfig(
        dataset_text_field="text",

        # Sequence handling
        max_seq_length=2048,
        packing=False,

        # Batching
        per_device_train_batch_size=16,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,

        # Training duration
        num_train_epochs=1,

        dataloader_num_workers=8,

        # Optimization
        learning_rate=1e-4,
        warmup_ratio=0.03,

        optim="adamw_torch_fused",

        weight_decay=0.01,

        lr_scheduler_type="cosine",

        # Logging
        logging_steps=10,

        # Evaluation
        eval_strategy="steps",
        eval_steps=25,
        do_eval=True,

        # Checkpoint saving
        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,

        # Best checkpoint loading
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # Precision
        bf16=True,

        # Misc
        seed=3407,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/7200 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/800 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [26]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,200 | Num Epochs = 1 | Total steps = 225
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Step,Training Loss,Validation Loss
25,0.557900,0.680241
50,0.669400,0.644271
75,0.561500,0.620499
100,0.651700,0.605316
125,0.592600,0.593231
150,0.698600,0.580670
175,0.479300,0.574236
200,0.610200,0.570956
225,0.545800,0.570197


In [3]:
model.save_pretrained_merged(
    "llama31-email-security-16bit",
    tokenizer,
    save_method = "merged_16bit",
)

NameError: name 'model' is not defined

In [ ]:
from huggingface_hub import login

login("")

In [10]:
!pip install -q huggingface_hub
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(

    folder_path = "llama31-email-security-16bit",

    repo_id = "kugu/llama-8b-instruct-email-classify",

    repo_type = "model",
)


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/kugu/llama-8b-instruct-email-classify/commit/7b8870da0d46a913f58c3a496865661123583fdc', commit_message='Upload folder using huggingface_hub', commit_description='', oid='7b8870da0d46a913f58c3a496865661123583fdc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kugu/llama-8b-instruct-email-classify', endpoint='https://huggingface.co', repo_type='model', repo_id='kugu/llama-8b-instruct-email-classify'), pr_revision=None, pr_num=None)